# 10 - Reading the offset state

**Purpose.** To explain what session 04 found: what was captured, how a three-hour bias run
decided between two mechanisms for the black-level state, why that verdict reaches back into
constants published a week earlier, and what the project must do next because of it. `09` is the
notebook that *made* these numbers, and is written for someone checking the work. This one is
written for someone deciding what to do next.

**What it is not for.** It measures nothing and writes nothing. Every number is read back from
`results/` - `offset_state_constants.json`, `offset_state_frames.csv`,
`offset_state_settings.csv`, and the earlier sessions' `bias_constants.json`,
`ptc_constants.json`, `dark_constants.json` and `pedestal_drift.csv`. Where arithmetic is done
below it is done on published numbers, to show what a published number is worth; if any of it
disagreed with `results/`, `results/` would be right and this notebook would be the bug.

**It assumes `00_statistics.ipynb`.** Why a plane mean over a quarter of a million pixels resolves
a hundredth of a count, why a pair difference measures a width and not a level, what a
point-biserial correlation is - all of that is explained there, on these same conventions, and is
cited rather than re-derived.

**The headline.** Session 03 found the black level sitting in discrete states about one count
apart and could not say why, because it varied nothing that might cause it. Session 04 varied
gain, offset and reconfiguration, against two predictions written down before the frames existed.
The answer is **H2: the state happens before the gain stage**. The step is
**0.158 counts at gain 0 and 10.31 at gain 450** - a factor of 65 - where a post-ADC level would
have been the same 0.99 counts at both.

Four things are worth your attention:

1. the verdict is **not close**: RMS residual 0.267 counts for H2 against 6.613 for H1, on a
   measurement whose precision on a plane mean is a few thousandths of a count;
2. it **reaches backwards**. Session 01's published pedestal at gain 450 is a mixture mean of a
   distribution nobody knew was bimodal, and session 02's `g` inherits that through the PTC's
   signal axis. `R` is safe, and the reason it is safe is worth understanding;
3. the strongest evidence is **out of sample and cost nothing to collect**: H2 predicts a
   0.488-count step at gain 100, which this night could not resolve, and session 01's
   `pedestal_drift.csv` - 450 frames shot a week earlier for a different purpose - is starkly
   bimodal at that gain;
4. **in electrons it is 0.5 to 1.5 e-**, against a sky delivering hundreds per sub. The state is a
   calibration problem, not an imaging problem, and MISSION's model is untouched.

In [ ]:
import json
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from astropix import stats as ST

pd.set_option("display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "font.size": 8})

RESULTS = pathlib.Path("..") / "results"

fr = pd.read_csv(RESULTS / "offset_state_frames.csv")
settings = pd.read_csv(RESULTS / "offset_state_settings.csv")
drift = pd.read_csv(RESULTS / "pedestal_drift.csv")
sweep = pd.read_csv(RESULTS / "bias_sweep.csv")
with open(RESULTS / "offset_state_constants.json") as fh:
    K = json.load(fh)
with open(RESULTS / "bias_constants.json") as fh:
    K1 = json.load(fh)
with open(RESULTS / "ptc_constants.json") as fh:
    K2 = json.load(fh)
with open(RESULTS / "dark_constants.json") as fh:
    K3 = json.load(fh)

GAIN, OFFSET = 250, K1["project_offset"]["value"]
HCG = K1["hcg_threshold_gain"]["value"]
PED_FIT = K1["pedestal_fit"]["value"]
PED_PER_OFFSET = K1["pedestal_per_offset_unit"]["value"]
G_E = {int(k): v for k, v in K2["system_gain"]["value"].items()}
ANCHOR = K3["offset_state_step"]["value"]          # session 03, 0.9931 counts at gain 250
SKY_E_PER_S = 1.594                                 # L32, green, unfiltered, Bortle 5-6

VERDICT = K["offset_state_mechanism"]["value"]
STEP = {int(k): v for k, v in K["offset_state_step_vs_gain"]["value"].items()}
STEP_E = {int(k): v for k, v in K["offset_state_step_electrons"]["value"].items()}
RECORD = K["session_record"]["value"]
RECONFIG = K["offset_state_reconfiguration"]["value"]
PLANES = ["R", "G1", "G2", "B"]


def predict_h1(gain):
    """A digital or post-ADC level: the same number of counts at every gain."""
    return ANCHOR


def predict_h2(gain):
    """An analog shift, riding session 01's own analog pedestal term B * 10**(gain/200),
    normalised so that both hypotheses pass through session 03's anchor at gain 250."""
    branch = PED_FIT["hcg" if gain >= HCG else "lcg"]
    at_anchor = PED_FIT["hcg"]["B"] * 10 ** (GAIN / 200)
    return ANCHOR * branch["B"] * 10 ** (gain / 200) / at_anchor


print(f"{K['offset_state_mechanism']['source_frames']} frames over {fr.t_min.max():.0f} min, "
      f"captured {K['offset_state_mechanism']['measured_on']}, "
      f"{RECORD['temp_range_C'][0]} to {RECORD['temp_range_C'][1]} C, "
      f"duty {RECORD['duty_range_pct'][0]}-{RECORD['duty_range_pct'][1]}%")
print(f"verdict: {VERDICT}")
print(f"anchor: session 03's {ANCHOR} counts at gain {GAIN}, which both predictions pass through")

---

## 1. What the night looked like

Three arms, and each one asks a different question of the same camera.

| arm | frames | what it varies | what it can see |
|---|---|---|---|
| A | 290 | nothing at all | onset, run lengths, whether the state tracks duty or idle gap |
| B | 420 (240 gain + 180 offset) | gain, then offset - interleaved and cycled, never blocked | the step against gain: the decisive test |
| C | 100 (80 + 20) | a no-op `configure`, and a close/reopen | H3, directly |

**Arm A is two thirds of the frames and nearly three quarters of the wall clock**, because its
gaps are drawn from {2, 15, 60} s and arm B's bias frames come back as fast as the camera can read
them. **All 35 far frames in the night are in arm B's gain blocks, and all of them at gain 0 or
gain 450** - the two extremes of the swept range. Nothing at gain 100 or 250, in 570 frames. Hold
that oddity: section 2 explains half of it and section 4 admits the other half is unexplained.

**Arm B is cycled rather than blocked, and that is session 03's mistake corrected.** That night's
exposure ladder ran short to long, so a state that depended on exposure and a state that depended
on elapsed time left identical traces. Here each gain is visited three times in a reshuffled
order, so a state that arrives mid-run contaminates one cycle and shows up as an inconsistency
between cycles rather than as a spurious setting effect.

**The gates all passed.** White balance proved in the pixels at every gain change and at every
reopen; the pedestal within 5 counts of where session 01 left it; and gate 3 - the first twenty
frames of arm A spreading 0.016 counts against a 0.1 limit - saying the camera was in one state at
minute zero, so the onset question was genuinely open rather than already answered.

The plot below is every frame's *departure from its own peer group*, not its level: levels at gain
450 sit 160 counts above levels at gain 0, and a plot of raw levels would show nothing but the
pedestal law.

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10.2, 5.0),
                       gridspec_kw={"height_ratios": [2, 1]})

ARMS = (("A", "o", "0.35", "A - nothing changes"),
        ("Bg", "s", "tab:blue", "B-gain - four gains, three cycles"),
        ("Bo", "s", "tab:purple", "B-offset - three offsets, three cycles"),
        ("Cs", "^", "tab:olive", "C-set - no-op reconfigure"),
        ("Co", "v", "tab:brown", "C-open - close and reopen"))
for arm, marker, colour, lab in ARMS:
    a = fr[fr.arm == arm]
    ax[0].plot(a.t_min, a.departure, marker, ms=3, lw=0, color=colour,
               alpha=0.7, label=lab)
f = fr[fr.far]
ax[0].plot(f.t_min, f.departure, "o", ms=5, mfc="none", color="crimson",
           label=f"far state ({len(f)} frames)")
ax[0].axhline(0, color="0.6", lw=0.6)
ax[0].set(ylabel="departure from peer-group median, counts",
          title="the night: every frame against wall clock, in its own peer group")
ax[0].legend(ncol=3, fontsize=7)

ax[1].plot(fr.t_min, fr.gain, ".", ms=2, color="tab:blue")
ax[1].set(xlabel="minutes into the run", ylabel="gain", yticks=[0, 100, 250, 450],
          title="the schedule: arm A flat, arm B cycled, arm C back at the baseline")
plt.tight_layout()

print(fr.groupby("arm").agg(frames=("far", "size"), far=("far", "sum"),
                            minutes=("t_min", lambda s: round(s.max() - s.min()))
                            ).to_string())
print(f"\ngate 3: first 20 frames of arm A spread "
      f"{RECORD['gate3_spread_counts']:.4f} counts against a "
      f"{RECORD['gate3_limit_counts']} limit - one state at minute zero")
print(f"cooled in {RECORD['cooldown_s'] / 60:.1f} min from "
      f"{RECORD['sensor_at_cooler_on_C']} C; equilibrated at power-on: "
      f"{RECORD['equilibrated_at_power_on']}; seed {RECORD['seed']}")

## 2. The decisive test: how the step scales with gain

Both hypotheses are anchored to the same measurement - session 03's 0.9931-count step at gain 250
- so neither gets to fit anything. What separates them is one exponent.

**H1, after the gain stage.** A digital or post-ADC level is added to a number that has already
been converted. Nothing downstream of the ADC knows what the analog gain was, so the step is the
same count everywhere.

**H2, before the gain stage.** A shift at the sense node, or in the analog reference, is
*amplified* on its way to the ADC by exactly the factor the gain stage applies. Session 01 already
measured that factor and published it as the analog half of the pedestal law,
`pedestal = A + B * 10**(gain/200)`, with separate `B` on either side of the HCG threshold. H2 is
that same curve, rescaled to pass through the anchor - a prediction with no free parameters.

The two differ by a factor of 6 at gain 0 and a factor of 10 at gain 450. A plane mean resolves a
few thousandths of a count. This was never going to be close.

**Two of the four gains resolved nothing, and that is not a zero.** `stats.offset_state` reports
`None` when the states are closer together than it can split them, and the honest reading of a
`None` is an upper bound at the resolution limit. Gain 100 is the interesting one: H2 predicts
0.488 counts against a 0.776-count limit, so it *cannot* resolve there even if H2 is right.
Section 7 is what happened when we looked for that step somewhere else.

In [ ]:
gain_arm = settings[(settings.offset == OFFSET)].sort_values("gain").copy()
gain_arm["h1"] = [predict_h1(g) for g in gain_arm.gain]
gain_arm["h2"] = [predict_h2(g) for g in gain_arm.gain]
gain_arm["resid_h1"] = gain_arm.separation - gain_arm.h1
gain_arm["resid_h2"] = gain_arm.separation - gain_arm.h2
res = gain_arm[gain_arm.separation.notna()]

fig, ax = plt.subplots(1, 2, figsize=(10.0, 3.4))

g = np.linspace(0, 460, 400)
ax[0].plot(g, [predict_h1(x) for x in g], "-", lw=1.0, color="tab:blue",
           label="H1: after the gain stage")
for lo, hi in ((0, HCG - 1), (HCG, 460)):
    seg = np.linspace(lo, hi, 200)
    ax[0].plot(seg, [predict_h2(x) for x in seg], "-", lw=1.0, color="crimson",
               label="H2: before the gain stage" if lo == 0 else None)
ax[0].plot(res.gain, res.separation, "o", ms=7, color="0.15", label="measured")
un = gain_arm[gain_arm.separation.isna()]
ax[0].plot(un.gain, un.resolution_limit, "v", ms=7, mfc="none", color="0.15",
           label="unresolved: upper bound")
ax[0].plot(GAIN, ANCHOR, "*", ms=12, color="tab:green", label="session 03 anchor")
ax[0].axvline(HCG, color="0.7", lw=0.8, ls=":")
ax[0].set(yscale="log", xlabel="gain", ylabel="step between states, ADC counts",
          title="one exponent separates the two mechanisms")
ax[0].legend(fontsize=6.5, loc="upper left")

ax[1].axhline(0, color="0.6", lw=0.6)
ax[1].plot(res.gain, res.resid_h1, "o", ms=7, color="tab:blue", label="H1 residual")
ax[1].plot(res.gain, res.resid_h2, "o", ms=7, color="crimson", label="H2 residual")
ax[1].set(xlabel="gain", ylabel="measured minus predicted, counts",
          title="residuals, on the two gains that resolved")
ax[1].legend(fontsize=7)
plt.tight_layout()

show = ["gain", "n", "separation", "scatter", "resolution_limit", "h1", "h2",
        "resid_h1", "resid_h2", "far", "occupancy"]
print(gain_arm[show].round(4).to_string(index=False))

rms = lambda x: float(np.sqrt(np.mean(np.square(x))))
print(f"\nRMS residual   H1 {rms(res.resid_h1):.4f}   H2 {rms(res.resid_h2):.4f} counts")
print(f"published verdict: {VERDICT} "
      f"(from {K['offset_state_mechanism']['source_frames']} frames, "
      f"{K['offset_state_mechanism']['notebook']})")
print(f"\nH2 is out by {100 * abs(res.resid_h2 / res.h2).max():.1f}% at worst; "
      f"H1 by {100 * abs(res.resid_h1 / res.h1).max():.0f}%")

## 3. Counts, electrons, and why that is a statement about the sense node

A step measured in counts is a statement about *nothing in particular* until it is converted:
counts are what the ADC printed, and the ADC sits at the end of a chain whose gain we chose. Push
the same step back through session 02's measured `g` and it becomes a number of **electrons**,
which is a statement about charge at the sense node - and charge is where physics happens.

The conversion turns a factor of 65 in counts into a factor of 3 in electrons. That remaining
factor of 3 is not noise, and it is not left unexplained: **the two resolved gains sit on opposite
sides of the HCG threshold**, where the sensor switches conversion gain, and session 01's own
pedestal law says the analog term changes by `B_hcg / B_lcg` = 0.362 across it. The measured ratio
of the step in electrons is 0.337. Those agree to 7%, on numbers from three different sessions.

So the most specific thing this night can say is: **the state is a roughly fixed packet of charge
at the sense node, about 1.5 e- in low conversion gain and 0.5 e- in high**, and everything else -
the factor of 65 in counts, the jump at gain 200 - is the amplifier chain faithfully doing its
job on it. It behaves like one electron's worth of reference, give or take.

**Which is why the model does not care.** A sub at these skies collects hundreds of electrons of
sky per pixel. Half an electron of black-level offset is 0.5% of a 60 s sub's sky signal and 0.05%
of a 600 s one - and it enters as an offset, not as a variance. **The state is a calibration
problem, not an imaging problem.** What it touches is the bench-measured constants MISSION's model
*consumes*, which is section 6.

In [ ]:
res_e = res.copy()
res_e["g_e"] = [G_E[int(x)] for x in res_e.gain]
res_e["step_e"] = res_e.separation * res_e.g_e
res_e["branch"] = np.where(res_e.gain >= HCG, "hcg", "lcg")

fig, ax = plt.subplots(1, 2, figsize=(9.8, 3.2))
ax[0].plot(res_e.gain, res_e.separation, "o-", ms=7, lw=1.0, color="crimson",
           label="in ADC counts")
ax[0].set(xlabel="gain", ylabel="step, ADC counts", yscale="log",
          title="the same state, in the unit the bench works in")
ax[0].legend(fontsize=7)
ax2 = ax[1]
ax2.plot(res_e.gain, res_e.step_e, "o-", ms=7, lw=1.0, color="tab:green",
         label="in electrons")
ax2.set(xlabel="gain", ylabel="step, e-", ylim=(0, 1.8),
        title="and in the unit the model works in")
ax2.axvline(HCG, color="0.7", lw=0.8, ls=":")
ax2.annotate("HCG threshold", (HCG, 1.6), fontsize=7, color="0.4",
             xytext=(6, 0), textcoords="offset points")
ax2.legend(fontsize=7)
plt.tight_layout()

print(res_e[["gain", "branch", "separation", "g_e", "step_e"]].round(4).to_string(index=False))
lo, hi = res_e.step_e.iloc[0], res_e.step_e.iloc[-1]
b_ratio = PED_FIT["hcg"]["B"] / PED_FIT["lcg"]["B"]
print(f"\nratio in counts:    {res_e.separation.iloc[-1] / res_e.separation.iloc[0]:.1f}x")
print(f"ratio in electrons: {hi / lo:.3f}")
print(f"session 01's B_hcg / B_lcg across the same threshold: {b_ratio:.3f} "
      f"({100 * (hi / lo / b_ratio - 1):+.0f}%)")

print("\nwhat half an electron is worth, against L32's sky:")
for t in (60, 300, 600):
    print(f"  {t:3d} s sub: sky {SKY_E_PER_S * t:7.1f} e-/px, "
          f"state {hi:.2f} e- = {100 * hi / (SKY_E_PER_S * t):.4f}% of it")

## 4. What the night did *not* see, and why three nulls are results

**H3 is refuted, directly and cheaply.** Forty pairs of frames straddling a `configure` call with
identical values, and ten straddling a full close and reopen with a re-cool in between: **zero
changed state**, against arm A's frame-to-frame baseline of zero. Reconfiguration does not trigger
it. That means *do not reconfigure mid-block* does **not** become a protocol rule - and a rule we
do not have to add is worth as much as one we do.

**Arm A saw nothing in 290 frames over two hours.** No far frames, so there are no run lengths, no
transition probabilities and no regression: `offset_state_transitions` and
`offset_state_regressors` publish `None` and say why. Duty was flat at 58-59% across the whole of
it, so session 03's suspicion that the state tracks a warming room is neither confirmed nor
refuted - there was no warm-up transient in this window to find one in.

**One confound this leaves, named rather than buried, and half of it breaks.** Cooler duty steps
from 58-59% to 60-62% at the arm A / arm B boundary and stays up: arm B's bias frames come
back-to-back with no gaps, a heavier thermal load than arm A's one frame per 26 seconds. So arm
A's 290-frame null sits entirely at 58-59%, and **every far frame in the night was taken at 60% or
above**. Arm A cannot clear duty; duty is confounded with the arm, and therefore with the idle gap
as well.

**What the night does break is the other half.** At a common 62% duty, gain 250 has 40 frames and
none of them far, at a 0.043-count resolution limit where a 0.993-count step would have shown at
23 sigma - while gains 0 and 450, at that same 62%, do go far. So duty does not explain the
pattern *across gains*, and nothing here touches the headline: the verdict rests on the **size**
of the step, and a duty effect would have to reproduce a factor of 65 to reach it. What stays open
is what governs *how often*, and the cell below is what that looks like laid out.

**The anchor gain showed nothing, and this is the session's own soft spot.** Gain 250 got 510
frames at a 0.043-count resolution limit, where session 03's 0.993-count step would have stood out
at 23 sigma. Zero far frames. Session 03 saw the state in 4.1% of 244 frames at that exact
setting, four days earlier. The step *sizes* fit H2 to a few percent and section 7 confirms the
law out of sample, so the mechanism call stands - but **the recurrence is condition-dependent in a
way nothing here explains**, and the gain the whole normalisation hangs on is the one gain this
night has no step of its own for.

**And the 4.32% occupancy is not a rate.** It is 33% at gain 0, 25% at gain 450, and zero at gains
100 and 250. An occupancy averaged over a gain sweep describes no setting anyone would use.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(10.4, 3.0))

a = fr[fr.arm == "A"]
ax[0].plot(a.t_min, a.departure, "o", ms=2.5, color="0.35")
ax[0].axhline(0, color="0.6", lw=0.6)
ax[0].axhspan(-ANCHOR / 2, ANCHOR / 2, color="crimson", alpha=0.10,
              label="half session 03's step")
ax[0].set(xlabel="minutes", ylabel="departure, counts",
          title=f"arm A: {len(a)} frames, {int(a.far.sum())} far")
ax[0].legend(fontsize=7)

lab = [f"gain {int(r.gain)}\noffset {int(r.offset)}" for r in settings.itertuples()]
ax[1].bar(lab, 100 * settings.occupancy,
          color=["crimson" if o else "0.7" for o in settings.occupancy])
ax[1].set(ylabel="% of frames in the far state",
          title="occupancy is a property of the setting")
ax[1].tick_params(axis="x", labelsize=6)

names = ["arm A\nframe to frame", "C-set\nno-op configure", "C-open\nclose and reopen"]
rates = [RECONFIG["arm_a_frame_to_frame"], RECONFIG["c_set"]["rate"],
         RECONFIG["c_open"]["rate"]]
ax[2].bar(names, rates, color=["0.5", "tab:blue", "tab:purple"])
ax[2].set(ylabel="transition rate per interval", ylim=(0, 1),
          title="H3 wanted the two right-hand bars\nwell above the left one")
ax[2].tick_params(axis="x", labelsize=6.5)
plt.tight_layout()

print(f"C-set:  {RECONFIG['c_set']['changed']} of {RECONFIG['c_set']['pairs']} pairs changed state")
print(f"C-open: {RECONFIG['c_open']['changed']} of {RECONFIG['c_open']['pairs']} pairs changed state")
print(f"arm A:  {int(a.far.sum())} of {len(a)} frames far, duty "
      f"{a.duty_pct.min()}-{a.duty_pct.max()}%, temperature "
      f"{a.ccd_temp.min()} to {a.ccd_temp.max()} C")
print(f"\nregressors (point-biserial r, arm A): "
      f"{K['offset_state_regressors']['value']}")
print("all None: there is no far/near label to regress against in arm A")
print(f"\noccupancy, whole session: {100 * K['offset_state_occupancy']['value']:.2f}% "
      f"- an average over settings that behave completely differently:")
print(settings[["gain", "offset", "n", "far", "occupancy"]].to_string(index=False))

# Duty against gain, since the two are not independent in this night: arm A ran
# cool and quiet, arm B ran back-to-back and warm.  What breaks the confound is
# a gain that saw no state at the *same* duty as the gains that did.
print("\nframes by gain and cooler duty (far / total), offset 15:")
sub = fr[fr.offset == OFFSET]
tab = sub.pivot_table(index="gain", columns="duty_pct", values="far",
                      aggfunc=["sum", "size"]).fillna(0).astype(int)
grid = pd.DataFrame({d: [f"{tab[('sum', d)][g]}/{tab[('size', d)][g]}"
                         if tab[('size', d)][g] else "-"
                         for g in tab.index]
                     for d in sorted(sub.duty_pct.unique())}, index=tab.index)
print(grid.to_string())
print("gain 250 at 62% duty is the cell that matters: 40 frames, 0 far, at a "
      "resolution limit\nof 0.0435 counts - the same duty at which gains 0 and "
      "450 do go far.")

## 5. The threshold session 03 used had to go, and the night refutes it rather than arguing with it

Session 03 called a frame *far* when it sat more than **0.5 counts** from its peers. That rule is
correct exactly when the step is 0.993 counts everywhere - which is H1, the hypothesis this night
rejected. Protocol rule 1 anticipated this and replaced it before the frames existed: a frame is
far when its departure exceeds **half the modal separation of its own peer group**, floored at
five times that group's within-state scatter. Both numbers are measured from the group, so the
threshold cannot assume the answer.

The published `offset_state_rule_agreement` is **false**, and gain 0 is where it breaks: the step
there is 0.158 counts, so the old rule found **0** far frames where the new one found **20 of
60**. Those twenty frames are not marginal - the two states at gain 0 are separated by 77 times
the within-state scatter of a plane mean. They are unmistakable, and a fixed threshold calibrated
at a different gain simply could not see them.

**This is the general shape of the lesson.** A threshold in counts is a threshold in the wrong
unit for a mechanism that lives before the gain stage. The replacement is per-setting because the
thing being detected is per-setting.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9.8, 3.2))

s = settings[settings.offset == OFFSET].sort_values("gain")
ax[0].plot(s.gain, s.threshold, "o-", ms=6, lw=1.0, color="crimson",
           label="modal rule: half the separation, floored at 5x scatter")
ax[0].axhline(0.5, color="tab:blue", lw=1.0, ls="--",
              label="session 03's fixed 0.5 counts")
ax[0].plot(s.gain, s.resolution_limit, "^-", ms=5, lw=0.8, color="0.5",
           label="resolution limit (3x scatter)")
ax[0].set(yscale="log", xlabel="gain", ylabel="ADC counts",
          title="the threshold has to move with the step")
ax[0].legend(fontsize=6.5)

g0 = fr[(fr.gain == 0) & (fr.offset == OFFSET)]
ax[1].hist(g0.level, bins=40, color="0.6")
sep0 = float(settings.loc[(settings.gain == 0), "separation"].iloc[0])
sc0 = float(settings.loc[(settings.gain == 0), "scatter"].iloc[0])
ax[1].set(xlabel="plane mean, ADC counts", ylabel="frames",
          title=f"gain 0: two states {sep0:.3f} counts apart\n"
                f"= {sep0 / sc0:.0f} sigma, and 0.5 counts saw neither")
plt.tight_layout()

print(settings[["gain", "offset", "separation", "scatter", "threshold", "far",
                "far_session03_rule", "agrees_with_session03"]].round(4).to_string(index=False))
print(f"\npublished rule agreement: {K['offset_state_rule_agreement']['value']}")
print(f"at gain 0 the separation is {sep0 / sc0:.0f}x the within-state scatter - "
      f"not a marginal call, just an invisible one at a 0.5-count threshold")
print("\nall four planes step together, which is what makes it a level and not a colour:")
print(settings[["gain", "offset"] + [f"step_{p}" for p in PLANES]]
      .round(4).to_string(index=False))

## 6. What it costs each published constant, taken one at a time

"High gain is in question" was the first reading, and it was too coarse - it sent the first pass
after the wrong constant. Each constant has to be checked against *how it was actually computed*.

| constant | verdict | why |
|---|---|---|
| `pedestal` | **hit** | measured as a level, and a level is exactly what moves |
| `R(gain)` | **safe** | pair-difference sigma over root two: a *spatial width*, and a uniform level shift does not change a width |
| `g(gain)` | **exposed** | the PTC's signal axis is `mean - pedestal`, and its intercept is *fixed* at R-squared rather than fitted, so a shifted signal axis lands entirely in the slope |
| `R` in electrons | hit by inheritance | `R_e = R_ADU * g` |
| `ceiling(gain)` | untouched | 2 counts against 4095 is 0.05%, well under the +/-5% of reading a vendor chart |
| `D` | already right | session 03 published a bound and not a value *because* of this state |
| `eta_comb`, `t_dead`, `F_sky` | untouched | differences, timestamps, and an on-sky quantity where 0.1 e- is nothing |

**The pedestal case is demonstrable rather than argued.** Session 01 published 234.2696 counts at
gain 450, offset 15. Tonight the same setting has two states, and the published value is neither
of them: it sits between them, where the mean of a mixture lands. Back out the occupancy that
would put it exactly there and you get **20.0% far in session 01's frames**, against 25.0%
measured tonight - two nights, four days apart, agreeing on a quantity neither of them set out to
measure. **Gain 0 says the same thing more quietly**: its published pedestal implies 28.2% far
against 33.3% measured tonight, on a step of 0.158 counts - true, and worth nothing, because a
sixth of a count is beneath anything that consumes it.

**The `R` case is worth understanding because it is the one that keeps being got wrong.** The
intuition "high gain amplifies the state, so read noise is inflated" is wrong for the same reason
the glow gradient survived session 03: a quantity defined as a difference *in space* is blind to a
level that is uniform *across space*. Subtract two frames and the state cancels into the mean of
the difference image, which `R` never looks at.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9.8, 3.2))

for i, g in enumerate((450, 0)):
    grp = fr[(fr.gain == g) & (fr.offset == OFFSET)]
    pub = float(sweep[(sweep.gain == g) & (sweep.offset == OFFSET)].pedestal.iloc[0])
    near = grp[~grp.far].level.mean()
    far_l = grp[grp.far].level.mean()
    ax[i].hist(grp.level, bins=50, color="0.6")
    ax[i].axvline(pub, color="crimson", lw=1.2,
                  label=f"session 01 published {pub:.4f}")
    ax[i].axvline(near, color="tab:blue", lw=0.9, ls="--", label=f"near {near:.4f}")
    ax[i].axvline(far_l, color="tab:green", lw=0.9, ls="--", label=f"far {far_l:.4f}")
    ax[i].set(xlabel="plane mean, ADC counts", ylabel="frames",
              title=f"gain {g}: the published pedestal is a mixture mean")
    ax[i].legend(fontsize=6.5)
plt.tight_layout()

rows = []
for g in (0, 450):
    grp = fr[(fr.gain == g) & (fr.offset == OFFSET)]
    pub = float(sweep[(sweep.gain == g) & (sweep.offset == OFFSET)].pedestal.iloc[0])
    near, far_l = grp[~grp.far].level.mean(), grp[grp.far].level.mean()
    rows.append({"gain": g, "published": pub, "near": near, "far": far_l,
                 "step": near - far_l,
                 "implied occupancy in session 01": (near - pub) / (near - far_l),
                 "measured tonight": grp.far.mean()})
print(pd.DataFrame(rows).round(4).to_string(index=False))
print("\nread noise, published by session 01 at these gains, and why it is safe:")
print(sweep[(sweep.gain.isin([0, 450])) & (sweep.offset == OFFSET)]
      [["gain", "pedestal", "R_sd", "R_err", "plane_spread"]].round(4).to_string(index=False))
print("R_sd is a pair-difference width.  A uniform level shift moves the mean of a "
      "difference\nimage and not its sigma, so R in counts is immune - the same "
      "argument that saved\nthe glow gradient in session 03.")

## 7. The strongest evidence was already on disk

H2 makes a prediction this night could not test: a step of **0.488 counts at gain 100**, sitting
under that setting's 0.776-count resolution limit. The classifier resolved nothing there and said
so. But it left a clue - the frame-to-frame scatter at gain 100 is **0.2587 counts against 0.0145
at gain 250**, eighteen times larger, and an unresolved two-state mixture with a 0.488-count step
predicts a spread of about 0.24. The scatter says *hopping, unresolved* where the classifier says
nothing.

Testing that needed no camera. `results/pedestal_drift.csv` is session 01's **450 bias frames at
gain 100** over fifteen minutes, captured 2026-08-28 to ask an entirely different question: does
the pedestal drift? Run the same published classifier over that column and it splits cleanly into
two states.

**This is out of sample in three ways at once** - a different night, a different notebook, and a
gain this night's own classifier could not resolve. The in-sample fit *chose* H2 over H1; this
*predicted* an unobserved separation before anyone looked.

**It also explains a published uncertainty rather than merely adding to it.**
`pedestal_drift_rate` reads -0.00133 +/- 0.254 counts/min. That plus-or-minus is not measurement
scatter, it is the mixture width. The slope really is flat - a state hopping frame to frame
contributes spread and no slope at all - which is exactly how that analysis concluded "nothing in
the camera drifts on this timescale" while looking at a camera that was switching states
throughout.

**This section is a lead, not a harvest.** L31 - gain 100 not being repeatable while gain 200 was
- now has a named mechanism and a cheap test, and that is not the same as being verified. Nothing
enters `results/` except through a notebook that publishes it with provenance, and this notebook
publishes nothing. The numbers below are read back to show the shape of the thing; the session
that drains L31 is the one that writes them down.

In [ ]:
st = ST.offset_state(drift.pedestal.values)
sep = st["separation"]
h2_100 = predict_h2(100)
pub100 = float(sweep[(sweep.gain == 100) & (sweep.offset == OFFSET)].pedestal.iloc[0])

fig, ax = plt.subplots(1, 2, figsize=(9.8, 3.2))
ax[0].hist(drift.pedestal, bins=60, color="0.6")
ax[0].axvline(pub100, color="crimson", lw=1.2,
              label=f"session 01 published {pub100:.4f}")
ax[0].set(xlabel="pedestal, ADC counts", ylabel="frames",
          title=f"session 01's gain-100 drift run: {len(drift)} frames,\n"
                f"shot to ask whether the pedestal drifts")
ax[0].legend(fontsize=6.5)

ax[1].plot(drift.elapsed_s / 60, drift.pedestal, "o", ms=2.5, color="0.35")
ax[1].set(xlabel="minutes", ylabel="pedestal, ADC counts",
          title="the same frames against time: no slope, two levels")
plt.tight_layout()

print(f"two states, separated by {sep:.4f} counts")
print(f"H2 predicted {h2_100:.4f} at gain 100 "
      f"({100 * (sep / h2_100 - 1):+.1f}%), from a law normalised at gain 250 "
      f"on a different night")
print(f"H1 predicted {predict_h1(100):.4f}")
print(f"within-state scatter {st['scatter']:.4f}, occupancy {st['far'].mean():.3f}, "
      f"largest departure {st['worst_steps']:.2f} steps - no third state")
print(f"\nthe published drift rate is "
      f"{K1['pedestal_drift_rate']['value']} +/- "
      f"{K1['pedestal_drift_rate']['uncertainty']} counts/min; the mixture width "
      f"here is {drift.pedestal.std(ddof=1):.4f} counts,")
print("which is that uncertainty, and the reason the slope was correctly flat.")
print(f"\nand session 01's own published pedestal at gain 100, {pub100:.4f}, is "
      f"the mixture mean {drift.pedestal.mean():.4f} to "
      f"{abs(pub100 - drift.pedestal.mean()):.4f} counts")

## 8. What the session settled, and what it did not

**Settled, and available to every later notebook:**

| constant | value | what it unlocks |
|---|---|---|
| `offset_state_mechanism` | `H2` - before the gain stage | the state has a location in the chain, so its effect on any constant is predictable rather than feared |
| `offset_state_step_vs_gain` | 0.158 counts at gain 0, 10.31 at gain 450 | the size to expect at any gain, from a law with no free parameters |
| `offset_state_step_electrons` | 1.49 e- (LCG), 0.50 e- (HCG) | the unit the model works in, where the state is negligible |
| `offset_state_reconfiguration` | 0 of 50 pairs | H3 refuted; no protocol rule about reconfiguring |
| `offset_state_occupancy` | 4.3% overall, 33% / 25% / 0 / 0 by setting | the rejection cost a future sweep must budget for |
| `offset_state_rule_agreement` | `false` | session 03's fixed threshold is retired, with the frames that retire it |

**And three things that are not constants but are results:**

- **the night held**: 810 frames, 2.79 hours, every block at the setpoint, gate 1 passing at every
  gain change and every reopen;
- **arm A is a null**, and its emptiness is what makes the three-hour investment legible: no far
  frames, no transitions, no regressors, in 290 frames at a fixed configuration. It is not a null
  *in duty*, and section 4 says why - it ran at 58-59% where every far frame in the night sat at
  60% or above;
- **the state reached backwards into published work**, and only the constants defined as
  *differences in space* survived it untouched. That is now twice - session 03's glow gradient and
  DSNU, session 04's `R` - that the same design property has paid.

**Not settled, and worth being explicit about:**

- **why the state exists at all**, and what makes it recur. Gain 250 hopped for session 03 and sat
  still for 510 frames here, four days later, at the same setting. Nothing measured explains that,
  and cooler duty - the one regressor session 03 pointed at - is confounded with the arm in this
  night's design, so it is neither cleared nor convicted;
- **the anchor**. Both predictions were normalised through session 03's step at gain 250, and this
  night resolved no step there. Gain 200 - unmeasured, predicted at 0.5585 counts, and the HCG
  boundary itself - is the one point that would put the law on its own feet;
- **how exposed `g` really is**, which is the next session's whole job. It depends on how
  `05_ptc.ipynb` took its pedestal per rung: a systematic ~2-count offset at gain 450 is one
  thing, per-rung bias frames independently hopping +/-10 counts is another and worse. That check
  reads existing frames and needs no camera;
- **the 300 s / 600 s confound** from session 03, which H2 puts back in scope. At gain 250 it was
  a one-count question not worth a night; at gain 450 the same effect is ten counts, and it is
  answerable in far less than a night;
- **L31**, which section 7 gives a mechanism and a cheap test, and which stays in the queue until
  a notebook publishes the bimodality with provenance. A lead is not a harvest.

**What to do next.** Re-analyse, do not re-shoot. The frames that would settle `g` are already on
disk, and so are the ones that would drain L31. The camera comes back out for gain 200, and for
nothing else this session found.